# 01 · TabFormer, the temporal split, and a raw baseline

**NVIDIA TFM Blueprint, notebook 1 of 5.** A transaction foundation model reads
a customer's transactions the way a language model reads text: each purchase
becomes a few tokens, and the model is pretrained to predict the next one. No
fraud labels are used for that. The pretrained model then turns a customer's
recent history into an embedding, a vector that downstream models such as a
fraud classifier can use. This series follows the public
[NVIDIA TFM Blueprint](https://github.com/NVIDIA-AI-Blueprints/transaction-foundation-model)
(v1) step by step, rebuilt on this kit:

| Notebook | You learn | v1 notebook |
|---|---|---|
| 01 | The TabFormer book, the temporal split, and the raw baseline to beat | `01_dataset_baseline` |
| 02 | How a transaction becomes 12 tokens and a customer's history becomes one row | `02_seq_preproc_tokenization` |
| 03 | Pretraining by next-token prediction, and the label bug v1 shipped | `03_foundation_model_training` |
| 04 | Embedding each transaction with its history, and v1's embedding without it | `04_inference_embedding_extraction` |
| 05 | Whether the embeddings help a fraud model | `05_xgboost_fraud_detection` |

Every notebook runs on CPU in about a minute on a small generated book, so
you can read the whole series without downloading anything. To see real rows,
build the TabFormer splits once (notebook 01 shows how) and set
`TFM_TABFORMER_ROOT`; the notebooks then load 24 TabFormer users with
`config.small.yaml`. The full-size runs are recipe commands, listed in notebooks 03
and 05, not notebook cells.

By the end of this notebook you will know what one TabFormer transaction looks
like, how v1 splits the book in time, which question the fraud model answers,
and the baseline the foundation model has to beat.

## Getting TabFormer

[IBM TabFormer](https://github.com/IBM/TabFormer) is a public, synthetic book
of 24,386,900 card transactions by 2,000 users from 1991 to 2020, with a
fraud label on each transaction. About 0.12% of transactions are fraud.
Download `transactions.tgz` from the IBM repository's data link, then run:

```bash
uv run tfm prepare-tabformer --root ~/data/TabFormer
export TFM_TABFORMER_ROOT=~/data/TabFormer
```

This takes about 30 seconds on CPU. It follows v1's notebook 01: sort the book
by date and cut it where the running row count reaches 80% and 90%.

| partition | rows | fraud | dates |
|---|---|---|---|
| train | 19,508,123 | 24,924 | up to 2017-05-13 |
| val | 2,435,982 | 2,109 | 2017-05-14 to 2018-10-11 |
| test | 2,442,795 | 2,724 | from 2018-10-12 |

The split is by date, not by user: a user's early transactions land in train
and later ones in validation or test. 1,584 of the 2,000 users have training
rows. Every scored test transaction is dated after every training transaction.

In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

import os

import matplotlib.pyplot as plt
import pandas as pd

from tfm.config import load_recipe_config
from tfm.data.adapters.paths import ADAPTER_ENV
from tfm.extras.capability import extra_available
from tfm.runtime import prepare_data
from tfm.runtime.common import recipe_dir
from tfm.training.reproducibility import seed_everything

USING_TABFORMER = bool(os.environ.get(ADAPTER_ENV["tabformer"]))
CONFIG = recipe_dir("blueprint_v1") / ("config.small.yaml" if USING_TABFORMER else "config.smoke.yaml")
config = load_recipe_config(CONFIG)
if config.model.backbone == "huggingface" and not extra_available("transformers"):
    print("transformers extra not installed; using the eager decoder (uv sync --extra huggingface)")
    composition = config.model.composition.model_copy(update={"position": "nope"})
    model_cfg = config.model.model_copy(update={"backbone": "eager", "composition": composition})
    config = config.model_copy(update={"model": model_cfg})
seed_everything(config.seed)
prepared = prepare_data(config)
tokenizer = prepared.tokenizer
train_batch, val_batch, test_batch = prepared.batches
print("config", CONFIG.name, "| adapter", config.data.adapter, "| backbone", config.model.backbone)

## 1. One transaction

The adapter maps TabFormer's columns (`User`, `Amount`, `Merchant Name`,
`Is Fraud?`, ...) into the kit's `TransactionEvent`. The fraud label is kept
apart from the model inputs and carries the time it becomes known
(`available_at`), so a label can't leak into a history that ends before it.

In [ ]:
events = prepared.events
sample = events[0]
print("entity", sample.entity_id, "at", sample.event_timestamp.isoformat())
print("categorical", dict(sample.categorical))
print("numeric", dict(sample.numeric))
label = sample.labels["fraud"]
print("fraud label", label.value, "known at", label.available_at.isoformat())
print("events", len(events), "users", len({e.entity_id for e in events}))

## 2. The three partitions

On the generated book the counts are tiny, and its fraud rate is far above
TabFormer's 0.12%. The shape of the split is what matters here: train, then
validation, then test, in time order.

In [ ]:
def describe(name, partition):
    fraud = [e.labels["fraud"].value for e in partition if "fraud" in e.labels]
    return {
        "partition": name,
        "events": len(partition),
        "users": len({e.entity_id for e in partition}),
        "fraud rate": sum(fraud) / max(len(fraud), 1),
        "first": min(e.event_timestamp for e in partition).date(),
        "last": max(e.event_timestamp for e in partition).date(),
    }


splits = prepared.splits
summary = pd.DataFrame(
    [describe("train", splits.train), describe("validation", splits.validation), describe("test", splits.test)]
)
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].bar(summary["partition"], summary["events"], color="#76B900")
axes[0].set_title("Events per partition")
axes[1].bar(summary["partition"], summary["fraud rate"], color="#4C78A8")
axes[1].set_title("Fraud rate")
fig.tight_layout()
plt.show()
summary

## 3. The question: is this transaction fraud?

A fraud team scores transactions, so each row here is one labelled
transaction. Its features may use everything the customer did up to and
including it, but nothing after. Training rows are transactions from the
training period; evaluation rows are transactions from the test period.

Two more rules keep the score honest:

- **Different customers on each side.** A customer's rows are all training rows
  or all evaluation rows, never both, so a model can't score well by
  remembering customers. Customers are split in proportion to how many had
  fraud in the test period.
- **Fraud is rare, so the training side is rebalanced and the test side is not.**
  Training keeps every fraud and samples a fixed number of legitimate
  transactions per fraud (`history_window.train_negatives_per_positive`). The
  test side is a uniform sample, so it keeps the real fraud rate.

Notebooks 04 and 05 use exactly these rows.

In [ ]:
from tfm.runtime.evaluate import event_entity_split, history_window_tasks

split = event_entity_split(prepared, config.seed)
tasks = history_window_tasks(prepared, config, split)
rows, control = tasks["history_window"], tasks["single_event"]

pd.DataFrame(
    {
        "rows": [len(rows.y_train), len(rows.y_eval)],
        "fraud": [int(rows.y_train.sum()), int(rows.y_eval.sum())],
        "customers": [len({s.entity_id for s in rows.train_sequences}), len({s.entity_id for s in rows.eval_sequences})],
    },
    index=["train", "test"],
)

## 4. The raw baseline

Before any foundation model, fit a classifier on the transaction's own fields:
amount, hour, weekday, merchant category, chip use, state, and so on. This is
v1's raw arm and the number the embeddings have to improve on.

Identifiers are left out. v1's raw features included the `User` id, with the
same users on both sides of its split, so part of its baseline was each user's
remembered fraud rate. Here the user id is excluded
(`evaluation.tabular_baselines.exclude_keys`), and the customers differ between
sides anyway.

Average precision (AP, reported as `pr_auc`) is the main metric. With 0.12%
fraud, a model can rank almost every fraud above almost every legitimate
transaction (ROC-AUC 0.99) and still flag mostly legitimate ones at any useful
threshold; AP measures that second part. The head is XGBoost when the extra is
installed, as in v1, and logistic regression otherwise.

In [ ]:
import numpy as np

from tfm.evaluation.fraud.baselines import fit_predict_probe
from tfm.evaluation.fraud.metrics import classification_metrics
from tfm.evaluation.tabular import TabularFeaturizer

profiles = {p.entity_id: p for p in prepared.profiles}
raw = TabularFeaturizer.fit(
    rows.train_sequences, profiles, top_k=config.evaluation.tabular_baselines.top_k,
    history=False, exclude_keys=config.evaluation.tabular_baselines.exclude_keys,
)
x_train = np.nan_to_num(raw.transform(rows.train_sequences, profiles))
x_test = np.nan_to_num(raw.transform(rows.eval_sequences, profiles))
print(len(raw.names), "raw features, e.g.", raw.names[:6])

head = "xgboost" if extra_available("xgboost") else "logistic_regression"
scores = fit_predict_probe(x_train, rows.y_train, x_test, seed=config.seed, head=head)
baseline = classification_metrics(rows.y_eval, scores, k=config.evaluation.precision_at_k)
print("head", head, "| test AP", round(baseline["pr_auc"], 4), "| ROC-AUC", round(baseline["roc_auc"], 4))

> **What changed from v1**
>
> - **Same split, built on CPU.** `tfm prepare-tabformer` uses pandas where v1
>   used cuDF. Its parquet files match the ones v1's notebook writes, row for
>   row.
> - **Labels carry a time.** v1 stored `Is Fraud?` as a plain column. Here each
>   label records when it becomes known, and a history only sees labels known
>   by its cutoff.
> - **No identifiers, different customers.** v1's raw arm used the user id, with
>   the same users in training and test. Here neither holds.
> - **A test set at the real fraud rate.** v1 scored a stratified sample; here
>   the test side is a uniform sample.

## Takeaways

- TabFormer is split by date: train, then validation, then test.
- The task is per transaction, with different customers on each side.
- The raw-feature baseline is the bar the foundation model has to clear.
- On a 0.12% fraud rate, read average precision first.

Code: `tfm.data.adapters.tabformer`, `tfm.data.sequence_builders.event_rows`,
`tfm.evaluation.tabular`. The generated book's numbers only show that the
wiring works.

Next: [02 · Tokens](02_seq_preproc_tokenization.ipynb).